Week 5 Support vector machines, the kernel trick, and regularization for support vector machines

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Helper function to convert NOAA damage strings to numbers
def parse_damage(val):
    if pd.isna(val):
        return 0.0
    val_str = str(val).strip().upper()
    if val_str.endswith('K'):
        return float(val_str[:-1]) * 1_000
    elif val_str.endswith('M'):
        return float(val_str[:-1]) * 1_000_000
    elif val_str.endswith('B'):
        return float(val_str[:-1]) * 1_000_000_000
    try:
        return float(val_str)
    except ValueError:
        return 0.0

# 1. Load dataset
print("Loading capstone_2023_merged.csv...")
df = pd.read_csv("capstone_2023_merged.csv", low_memory=False)

# 2. Parse property damage
if 'DAMAGE_PROPERTY' in df.columns:
    df['DAMAGE_PROPERTY_NUMERIC'] = df['DAMAGE_PROPERTY'].apply(parse_damage)

# 3. Create Binary Target ($10,000 claim threshold)
target_col = 'amountPaidOnBuildingClaim' if 'amountPaidOnBuildingClaim' in df.columns else 'amountPaidOnTotalClaim'
y_binary = (pd.to_numeric(df[target_col], errors='coerce').fillna(0) >= 10000).astype(int)

feature_cols = ['DAMAGE_PROPERTY_NUMERIC', 'primary_landcover_code', 'alternate_landcover_code']
feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols].fillna(df[feature_cols].median(numeric_only=True))

# 4. Train / Test Split & Standardization (Crucial for SVM distance calculations)
X_train, X_test, y_train, y_test = train_test_split(X, y_binary, test_size=0.2, random_state=42, stratify=y_binary)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Use a representative stratified sub-sample (15,000 rows) for efficient kernel optimization
X_sub, _, y_sub, _ = train_test_split(X_train_scaled, y_train, train_size=15000, random_state=42, stratify=y_train)

print("\n--- Training Week 5 Support Vector Machines ---")

# --- 1. Linear Kernel SVM ---
print("Fitting Linear SVM...")
svm_linear = SVC(kernel='linear', C=1.0, probability=True, random_state=42)
svm_linear.fit(X_sub, y_sub)
y_pred_lin = svm_linear.predict(X_test_scaled)
y_prob_lin = svm_linear.predict_proba(X_test_scaled)[:, 1]

# --- 2. Polynomial Kernel SVM (Degree = 2) ---
print("Fitting Polynomial SVM (degree=2)...")
svm_poly = SVC(kernel='poly', degree=2, C=1.0, probability=True, random_state=42)
svm_poly.fit(X_sub, y_sub)
y_pred_poly = svm_poly.predict(X_test_scaled)
y_prob_poly = svm_poly.predict_proba(X_test_scaled)[:, 1]

# --- 3. Radial Basis Function (RBF / Gaussian) Kernel SVM ---
print("Fitting Gaussian (RBF) SVM...")
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
svm_rbf.fit(X_sub, y_sub)
y_pred_rbf = svm_rbf.predict(X_test_scaled)
y_prob_rbf = svm_rbf.predict_proba(X_test_scaled)[:, 1]

# 5. Compile Comparison Metrics
results_w5 = pd.DataFrame({
    'Kernel Variant': ['Linear Kernel SVM', 'Polynomial Kernel (d=2)', 'Gaussian (RBF) Kernel'],
    'Accuracy': [accuracy_score(y_test, y_pred_lin), accuracy_score(y_test, y_pred_poly), accuracy_score(y_test, y_pred_rbf)],
    'Precision': [precision_score(y_test, y_pred_lin, zero_division=0), precision_score(y_test, y_pred_poly, zero_division=0), precision_score(y_test, y_pred_rbf, zero_division=0)],
    'Recall': [recall_score(y_test, y_pred_lin, zero_division=0), recall_score(y_test, y_pred_poly, zero_division=0), recall_score(y_test, y_pred_rbf, zero_division=0)],
    'F1-Score': [f1_score(y_test, y_pred_lin, zero_division=0), f1_score(y_test, y_pred_poly, zero_division=0), f1_score(y_test, y_pred_rbf, zero_division=0)],
    'ROC-AUC': [roc_auc_score(y_test, y_prob_lin), roc_auc_score(y_test, y_prob_poly), roc_auc_score(y_test, y_prob_rbf)]
})

print("\n=== Week 5 Support Vector Machine Results ===")
print(results_w5.to_string(index=False))

# 6. Save Comparison Table Image
fig, ax = plt.subplots(figsize=(12, 3), dpi=300)
ax.axis('off')

table = ax.table(
    cellText=np.round(results_w5.iloc[:, 1:].values, 4),
    rowLabels=results_w5['Kernel Variant'].values,
    colLabels=results_w5.columns[1:],
    cellLoc='center',
    loc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)

for (row, col), cell in table.get_celld().items():
    if row == 0 or col == -1:
        cell.set_facecolor('#40466e')
        cell.set_text_props(color='white', weight='bold')
    else:
        cell.set_facecolor('#f5f5f5' if row % 2 == 0 else 'white')

plt.savefig('week5_model_comparison.png', bbox_inches='tight', pad_inches=0.1)
plt.close()

print("\nSuccess! Saved Week 5 comparison table image as 'week5_model_comparison.png'")

Loading capstone_2023_merged.csv...

--- Training Week 5 Support Vector Machines ---
Fitting Linear SVM...
Fitting Polynomial SVM (degree=2)...
Fitting Gaussian (RBF) SVM...

=== Week 5 Support Vector Machine Results ===
         Kernel Variant  Accuracy  Precision   Recall  F1-Score  ROC-AUC
      Linear Kernel SVM  0.552696   0.552696 1.000000  0.711918 0.501315
Polynomial Kernel (d=2)  0.553319   0.553053 0.999795  0.712162 0.483190
  Gaussian (RBF) Kernel  0.552242   0.552666 0.996205  0.710929 0.509107

Success! Saved Week 5 comparison table image as 'week5_model_comparison.png'
